## Creating a Single Table from Multiple Snapshots

It is common to receive data as **periodic snapshots** — e.g. a termly extract of pupil records, a monthly export from a finance system, or a daily feed from an API. Each snapshot has the same structure but covers a different point in time.

To analyse trends or build a complete picture, you need to combine these into a single table.

In this notebook we’ll work with two **synthetic** school snapshots from `catalog_40_copper_analyst_training.messy_data`:
* `schools_autumn_2024` — 20 rows, columns include `school_urn`, `school_name`, `city`
* `schools_spring_2025` — 18 rows, but with **different column names** (`urn`, `establishment_name`, `local_authority`)

> **Note:** All data in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals. No real data has been used.

This is a realistic scenario: the same data provider changed their column naming between extracts.

### Key principles

* **Use `UNION ALL`**, not `UNION`. `UNION` silently removes duplicates, which hides data quality issues and is slower. Always prefer `UNION ALL` and handle duplicates explicitly.
* **Add a snapshot identifier.** Tag each extract so you always know which source a row came from. This is essential for debugging and for time-series analysis.
* **Validate column alignment.** Snapshots may drift over time — columns renamed, added, or removed. Check schemas match before combining.

### Watch out for

* **Overlapping snapshots** — if two extracts cover the same period, you’ll get genuine duplicates
* **Schema drift** — a column called `city` in one snapshot and `local_authority` in another
* **Changed grain** — one snapshot at pupil level, another at pupil-school level

In [0]:
-- Preview both snapshots side by side to spot schema differences
SELECT 'autumn' as snapshot, * FROM catalog_40_copper_analyst_training.messy_data.schools_autumn_2024 LIMIT 5;

In [0]:
SELECT 'spring' as snapshot, * FROM catalog_40_copper_analyst_training.messy_data.schools_spring_2025 LIMIT 5;

In [0]:
-- The column names differ between snapshots, so a naive UNION ALL would fail or misalign.
-- We must manually map columns to a common schema and tag the source snapshot.

CREATE OR REPLACE TEMP VIEW all_schools AS

SELECT
  school_urn
  ,school_name
  ,city as location
  ,school_type
  ,status
  ,phase
  ,metadata_json
  ,'Autumn 2024' as snapshot_term
FROM catalog_40_copper_analyst_training.messy_data.schools_autumn_2024

UNION ALL

SELECT
  urn as school_urn
  ,establishment_name as school_name
  ,local_authority as location
  ,establishment_type as school_type
  ,status
  ,phase_of_education as phase
  ,metadata_json
  ,'Spring 2025' as snapshot_term
FROM catalog_40_copper_analyst_training.messy_data.schools_spring_2025;

SELECT * FROM all_schools
ORDER BY school_urn, snapshot_term;

In [0]:
-- Validate: combined rows should equal the sum of individual snapshots
SELECT
  (SELECT COUNT(*) FROM catalog_40_copper_analyst_training.messy_data.schools_autumn_2024) as autumn_rows
  ,(SELECT COUNT(*) FROM catalog_40_copper_analyst_training.messy_data.schools_spring_2025) as spring_rows
  ,(SELECT COUNT(*) FROM all_schools) as combined_rows
  ,CASE
    WHEN (SELECT COUNT(*) FROM catalog_40_copper_analyst_training.messy_data.schools_autumn_2024)
       + (SELECT COUNT(*) FROM catalog_40_copper_analyst_training.messy_data.schools_spring_2025)
       = (SELECT COUNT(*) FROM all_schools)
    THEN 'Row counts match - UNION ALL is correct'
    ELSE 'Row count mismatch - investigate'
  END as validation;

### Going further: detecting schema drift dynamically

The manual approach above works when you already know which columns differ. But what if you’re combining dozens of snapshots and can’t visually inspect each one?

The `INFORMATION_SCHEMA.COLUMNS` view in Unity Catalog contains metadata about every column in every table. You can query it to **programmatically compare schemas** between two tables and surface any differences — renamed columns, added columns, type changes, or missing columns.

This is far more robust than eyeballing `SELECT *` and scales to any number of tables.

In [0]:
-- Use INFORMATION_SCHEMA to compare column names and types between two snapshots
-- This surfaces schema drift without needing to inspect the data manually

CREATE OR REPLACE TEMPORARY VIEW autumn_cols AS
  SELECT
    ordinal_position
    ,column_name
    ,data_type
  FROM catalog_40_copper_analyst_training.information_schema.columns
  WHERE table_schema = 'messy_data'
    AND table_name = 'schools_autumn_2024';

SELECT * FROM autumn_cols;

In [0]:
CREATE OR REPLACE TEMPORARY VIEW spring_cols AS 
SELECT
  ordinal_position
  ,column_name
  ,data_type
FROM catalog_40_copper_analyst_training.information_schema.columns
WHERE table_schema = 'messy_data'
  AND table_name = 'schools_spring_2025';

SELECT * FROM  spring_cols;

In [0]:
SELECT
  COALESCE(a.ordinal_position, s.ordinal_position) as position
  ,a.column_name as autumn_column
  ,s.column_name as spring_column
  ,a.data_type as autumn_type
  ,s.data_type as spring_type
  ,CASE
    WHEN a.column_name IS NULL          THEN 'New column in spring'
    WHEN s.column_name IS NULL          THEN 'Removed in spring'
    WHEN a.column_name != s.column_name THEN 'RENAMED'
    WHEN a.data_type   != s.data_type   THEN 'TYPE CHANGED'
    ELSE 'Match'
  END as drift_status
FROM autumn_cols a
FULL OUTER JOIN spring_cols s
  ON a.ordinal_position = s.ordinal_position
ORDER BY position;

### Building a dynamic column mapping

Once you’ve identified the drift, you can take it a step further: use SQL scripting with `EXECUTE IMMEDIATE` to **generate and run the UNION ALL query automatically**, based on the column metadata.

This avoids hard-coding column mappings and makes the process repeatable when new snapshots arrive. We’ll build this up in three steps:

1. **Create a column mapping** — pair each autumn column with its spring counterpart by ordinal position
2. **Generate the SQL string** — use the mapping to construct a valid `UNION ALL` query as a string
3. **Execute it dynamically** — run the generated query with `EXECUTE IMMEDIATE`

In [0]:
-- Step 1: Create a column mapping between the two snapshots
-- We pair columns by their ordinal position (1st column maps to 1st column, etc.)
-- The autumn column names become our 'target' (canonical) names
-- FULL OUTER JOIN ensures we keep columns even if they only exist in one snapshot

CREATE OR REPLACE TEMP VIEW col_mapping AS
SELECT
  COALESCE(a.ordinal_position, s.ordinal_position) as ordinal_position
  ,a.column_name as target_col
  ,s.column_name as source_col
  ,COALESCE(a.column_name, s.column_name) as unified_col
  ,CASE
    WHEN a.column_name IS NULL          THEN 'Added in spring'
    WHEN s.column_name IS NULL          THEN 'Removed in spring'
    WHEN a.column_name = s.column_name  THEN 'Same'
    ELSE 'Renamed'
  END as status
FROM autumn_cols a
FULL OUTER JOIN spring_cols s
  ON a.ordinal_position = s.ordinal_position
ORDER BY COALESCE(a.ordinal_position, s.ordinal_position);

SELECT * FROM col_mapping;

The mapping above shows that columns at the same ordinal position represent the same data — for example, position 1 maps `school_urn` (autumn) to `urn` (spring). Any row marked **"Renamed"** is where the column name changed between snapshots.

We use a `FULL OUTER JOIN` so that columns which only exist in one snapshot still appear in the mapping (marked **"Added in spring"** or **"Removed in spring"**). With our two tables, every column has a match — but this will handle schema changes cleanly if they occur.

### The query we want to generate

Our goal is to produce this query **automatically** from the mapping, rather than writing it by hand:

```sql
SELECT school_urn
      ,school_name
      ,city
      ,school_type
      ,status
      ,phase
      ,metadata_json
      ,'Autumn 2024' AS snapshot_term
FROM catalog_40_copper_analyst_training.messy_data.schools_autumn_2024
UNION ALL
SELECT urn AS school_urn
      ,establishment_name AS school_name
      ,local_authority AS city
      ,establishment_type AS school_type
      ,status
      ,phase_of_education AS phase
      ,metadata_json, 'Spring 2025' AS snapshot_term
FROM catalog_40_copper_analyst_training.messy_data.schools_spring_2025
```

To build this string from the mapping table, we need to:
1. Turn the `target_col` column into a **comma-separated list** of column names for the autumn half
2. Turn the spring column names into a comma-separated list, **renaming them to match the autumn names** where they differ (e.g. `urn AS school_urn`)
3. Wrap them in `SELECT ... FROM ... UNION ALL SELECT ... FROM ...`

The key functions we’ll use are `collect_list` and `aggregate` — let’s explore them one at a time.

In [0]:
-- collect_list gathers all values from a column into a single ARRAY
-- Here it turns the 7 rows of target_col into one array of 7 strings
SELECT collect_list(target_col) as column_names_array
FROM col_mapping;

The result is an **array** (a list) containing all seven column names:

```
["school_urn", "school_name", "city", "school_type", "status", "phase", "metadata_json"]
```

But we need a single string: `school_urn, school_name, city, ...`. That’s where `aggregate` comes in.

### What does `aggregate` do?

`aggregate` takes an array and reduces it to a single value by processing each element one at a time. It needs three arguments:

1. **The array** — the output of `collect_list`
2. **The starting value** — an empty string `''`
3. **A lambda function** — a rule that says: for each element, combine it with what we’ve built so far

The lambda `(acc, x) ->` gives names to two things:
* `acc` (accumulator) — the **running result** built up so far
* `x` — the **current element** from the array being processed

On the first pass, `acc` is the starting value (empty string). On each subsequent pass, `acc` is whatever the lambda returned last time.

In [0]:
-- aggregate reduces the array to a single comma-separated string
-- On each pass: if acc is empty, return x; otherwise return acc + ', ' + x
SELECT aggregate(
  collect_list(target_col)
  ,''                                                                 -- starting value
  ,(acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ', ', x) END -- lambda
) as autumn_column_list
FROM col_mapping;

Let’s trace through how `aggregate` processes the array step by step:

| Pass | `acc` (so far) | `x` (current) | Result |
| --- | --- | --- | --- |
| 1 | `''` (empty) | `school_urn` | `school_urn` |
| 2 | `school_urn` | `school_name` | `school_urn, school_name` |
| 3 | `school_urn, school_name` | `city` | `school_urn, school_name, city` |
| … | … | … | … |
| 7 | `school_urn, school_name, …, phase` | `metadata_json` | `school_urn, school_name, city, school_type, status, phase, metadata_json` |

That gives us the **autumn column list**. Now we need the spring equivalent — but with `AS` aliases for any renamed columns.

In [0]:
-- For the spring half, we need to alias renamed columns back to autumn names
-- e.g. 'urn AS school_urn' (renamed) vs just 'status' (unchanged)
SELECT aggregate(
  collect_list(
    CASE
      WHEN source_col = target_col THEN source_col             -- unchanged: just the column name
      ELSE concat(source_col, ' AS ', target_col)              -- renamed: source AS target
    END
  )
  ,''
  ,(acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ', ', x) END
) as spring_column_list
FROM col_mapping;

Now we have the two key pieces:

* **Autumn columns:** `school_urn, school_name, city, school_type, status, phase, metadata_json`
* **Spring columns:** `urn AS school_urn, establishment_name AS school_name, local_authority AS city, ...`

The final step is to wrap these into a complete SQL statement. We’ll store each piece in a **SQL variable** and then concatenate them together.

In [0]:
-- Step 2c: Store each component in a variable, then combine into the full query
--Declare the variable to hold the autumn column string
DECLARE OR REPLACE VARIABLE autumn_cols_str STRING;

-- Build the autumn column list
SET VAR autumn_cols_str = (
  SELECT aggregate(
    collect_list(target_col), '', (acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ', ', x) END
  )
  FROM col_mapping
);

SELECT autumn_cols_str as autumn_columns;

In [0]:
--Declare the variable to hold the spring column string
DECLARE OR REPLACE VARIABLE spring_cols_str STRING;

-- Build the spring column list (with aliases)
SET VAR spring_cols_str = (
  SELECT aggregate(
    collect_list(
      CASE WHEN source_col = target_col THEN source_col ELSE concat(source_col, ' AS ', target_col) END
    ), '', (acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ', ', x) END
  )
  FROM col_mapping
);

SELECT spring_cols_str as spring_columns;

We also need `char(39)` to produce a single-quote character (`'`) — this is needed to wrap `'Autumn 2024'` and `'Spring 2025'` as string literals inside the generated query.

In [0]:
-- Declare a variable to hold the query string
DECLARE OR REPLACE VARIABLE query STRING;
-- Declare a variable and set it to a single-quote
DECLARE OR REPLACE VARIABLE q STRING DEFAULT char(39);

-- Combine into the full query, wrapped in CREATE OR REPLACE TEMP VIEW
-- so the result is stored and can be queried afterwards
SET VAR query = concat(
  'CREATE OR REPLACE TEMP VIEW all_schools_dynamic AS '
  ,'SELECT ', autumn_cols_str, ', ', q, 'Autumn 2024', q, ' AS snapshot_term'
  ,' FROM catalog_40_copper_analyst_training.messy_data.schools_autumn_2024'
  ,' UNION ALL '
  ,'SELECT ', spring_cols_str, ', ', q, 'Spring 2025', q, ' AS snapshot_term'
  ,' FROM catalog_40_copper_analyst_training.messy_data.schools_spring_2025'
);

-- Display the generated query so we can inspect it before running
SELECT query as generated_sql;

The output above shows all three pieces built up step by step:

* `autumn_columns` — the simple column list for the first half of the UNION ALL
* `spring_columns` — the column list with renaming for the second half
* `generated_sql` — the complete query, wrapped in `CREATE OR REPLACE TEMP VIEW` so the result is saved for querying

This is exactly the same query we wrote by hand in cell 4 — but built automatically from the column metadata. Now we just need to execute it and then query the view.

In [0]:
-- Step 3: Execute the generated query to create the temp view
-- EXECUTE IMMEDIATE takes a SQL string and runs it as if you had typed it directly
EXECUTE IMMEDIATE query;

The view `all_schools_dynamic` has been created. We can now query it just like any other table — the data from both snapshots is combined, with column names aligned and a `snapshot_term` column identifying which extract each row came from.

In [0]:
-- Step 4: Query the dynamically created view
-- Preview the first few rows to confirm the structure looks correct
SELECT *
FROM all_schools_dynamic
ORDER BY school_urn, snapshot_term
LIMIT 10;

In [0]:
-- Summarise row counts by snapshot to confirm both extracts are present
SELECT
  snapshot_term
  ,COUNT(*) as row_count
FROM all_schools_dynamic
GROUP BY snapshot_term
ORDER BY snapshot_term;

### What if columns have been added or removed?

Our column mapping already uses a `FULL OUTER JOIN`, so it will preserve columns that only exist in one snapshot. But since both school tables happen to have the same column count, we haven’t seen this in action yet.

In practice, snapshots often gain or lose columns over time:
* A new field is added in a later extract (e.g. `pupil_premium_pct` appears in spring but not autumn)
* An old field is dropped or deprecated (e.g. `region` was in autumn but removed in spring)

For a `UNION ALL` to work, **both halves must have exactly the same columns in the same order**. So when a column is missing from one side, we need to fill the gap with `NULL`.

Let’s **simulate** an extra `pupil_premium_pct` column in spring and an extra `region` column in autumn, so we can see how the mapping and expression builder handle it.

In [0]:
-- Simulate an autumn snapshot with an extra column (region)
CREATE OR REPLACE TEMPORARY VIEW autumn_cols_test AS
SELECT * FROM autumn_cols
UNION ALL
SELECT 7 as ordinal_position, 'region' as column_name, 'STRING' as data_type;

-- Simulate a spring snapshot with a different extra column (pupil_premium_pct)
CREATE OR REPLACE TEMPORARY VIEW spring_cols_test AS
SELECT * FROM spring_cols
UNION ALL
SELECT 8 as ordinal_position, 'pupil_premium_pct' as column_name, 'DOUBLE' as data_type;

-- Updated column mapping using FULL OUTER JOIN
-- This preserves columns that exist in only one snapshot
CREATE OR REPLACE TEMP VIEW col_mapping_full AS
SELECT
  COALESCE(a.ordinal_position, s.ordinal_position) as ordinal_position
  ,a.column_name as target_col
  ,s.column_name as source_col
  ,COALESCE(a.column_name, s.column_name) as unified_col
  ,CASE
    WHEN a.column_name IS NULL THEN 'Added in spring'
    WHEN s.column_name IS NULL THEN 'Removed in spring'
    WHEN a.column_name = s.column_name THEN 'Same'
    ELSE 'Renamed'
  END as status
FROM autumn_cols_test a
FULL OUTER JOIN spring_cols_test s
  ON a.ordinal_position = s.ordinal_position
ORDER BY ordinal_position;

SELECT * FROM col_mapping_full;

Any row marked **"Added in spring"** has a `target_col` of `NULL` — it doesn’t exist in the autumn snapshot. Any row marked **"Removed in spring"** has a `source_col` of `NULL`.

For the `UNION ALL` to work, we need to handle each case:

| Scenario | Autumn SELECT | Spring SELECT |
| --- | --- | --- |
| **Same name** | `school_urn` | `school_urn` |
| **Renamed** | `school_name` | `establishment_name AS school_name` |
| **Removed in spring** | `phase` | `NULL AS phase` |
| **Added in spring** | `NULL AS pupil_premium_pct` | `pupil_premium_pct` |

The updated query builder below uses this logic to produce the correct `SELECT` expression for each side. You can see how `NULL AS pupil_premium_pct` appears in the autumn expression to fill the gap.

In [0]:
-- Build the autumn and spring column expressions, handling all four cases
DECLARE OR REPLACE VARIABLE autumn_cols_full STRING;
DECLARE OR REPLACE VARIABLE spring_cols_full STRING;

-- Autumn side: use the column if it exists, otherwise NULL AS unified_col
SET VAR autumn_cols_full = (
  SELECT aggregate(
    collect_list(
      CASE
        WHEN target_col IS NOT NULL THEN target_col           -- column exists in autumn
        ELSE concat('NULL AS ', unified_col)                  -- missing: fill with NULL
      END
    )
    ,''
    ,(acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ', ', x) END
  )
  FROM col_mapping_full
);

-- Spring side: use the column (with alias if renamed), or NULL if removed
SET VAR spring_cols_full = (
  SELECT aggregate(
    collect_list(
      CASE
        WHEN source_col IS NULL THEN concat('NULL AS ', unified_col)    -- removed: fill with NULL
        WHEN source_col = unified_col THEN source_col                  -- same name: no alias needed
        ELSE concat(source_col, ' AS ', unified_col)                   -- renamed: alias to unified name
      END
    )
    ,''
    ,(acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ', ', x) END
  )
  FROM col_mapping_full
);

SELECT
  autumn_cols_full as autumn_expression
  ,spring_cols_full as spring_expression;

The expressions confirm the logic works for all four cases:

* **Same / Renamed** columns appear as before
* **Removed in spring** — `region` appears in the autumn expression, and as `NULL AS region` in the spring expression
* **Added in spring** — `pupil_premium_pct` appears as `NULL AS pupil_premium_pct` in the autumn expression, and by name in the spring expression

We simulated the extra columns in the metadata only, so this particular query can’t be executed against our tables. But when working with real snapshots that genuinely have different columns, this `FULL OUTER JOIN` approach will produce a working query that fills every gap with `NULL` automatically.

### Scaling to any number of snapshots

Everything above still requires you to specify **which two tables** to compare. That's fine for two snapshots, but if you had ten terms of data — or fifty — you'd need to write a separate mapping for each pair.

We can make the **entire process** dynamic by letting SQL discover and combine all matching tables automatically:

1. **Discover** all matching tables in the schema
2. **Get columns** for every table from `INFORMATION_SCHEMA`
3. **Build a unified column list** using the first table as the reference
4. **Map** every table's columns to the unified list
5. **Generate a SELECT** for each table, filling gaps with `NULL`
6. **Combine** all SELECTs with `UNION ALL` and execute

In [0]:
-- Step 1: Discover all school snapshot tables in the schema
-- The LIKE pattern matches any table starting with 'schools_'
-- Add a new snapshot next term and it will be picked up automatically

CREATE OR REPLACE TEMP VIEW snapshot_tables AS
SELECT table_name
FROM catalog_40_copper_analyst_training.information_schema.tables
WHERE table_schema = 'messy_data'
  AND table_name LIKE 'schools_%'
ORDER BY table_name;

SELECT * FROM snapshot_tables;

In [0]:
-- Step 2: Get the columns for every snapshot table
-- This gives us one row per column per table

CREATE OR REPLACE TEMP VIEW snapshot_columns AS
SELECT table_name, ordinal_position, column_name
FROM catalog_40_copper_analyst_training.information_schema.columns
WHERE table_schema = 'messy_data'
  AND table_name IN (SELECT table_name FROM snapshot_tables)
ORDER BY table_name, ordinal_position;

SELECT * FROM snapshot_columns;

We now have a list of every table and every column. The next step is to decide on a single set of **canonical column names** that all tables will be mapped to.

The simplest approach: pick one table as the **reference** and use its column names. We'll use the first table alphabetically (`schools_autumn_2024`). Any column at the same ordinal position in another table will be renamed to match the reference.

In [0]:
-- Step 3: Build the unified column list from the reference table
-- The first table alphabetically becomes the reference
-- Its column names are the canonical names that all other tables map to

DECLARE OR REPLACE VARIABLE ref_table STRING;
SET VAR ref_table = (SELECT MIN(table_name) FROM snapshot_tables);

CREATE OR REPLACE TEMP VIEW unified_columns AS
SELECT ordinal_position, column_name as unified_col
FROM snapshot_columns
WHERE table_name = ref_table
ORDER BY ordinal_position;

SELECT * FROM unified_columns;

The unified column list contains the 7 columns from the reference table (`schools_autumn_2024`). Every other table's columns will be mapped to these names by ordinal position.

Now we need to check, for **every table** and **every unified column**: does that table have a column at the matching position? And if so, does it need renaming?

We do this by pairing every table with every unified column using a `CROSS JOIN`. A `CROSS JOIN` creates **all possible combinations** of rows from two tables — so if we have 2 tables and 7 unified columns, we get 14 rows (one per table per column). We then use a `LEFT JOIN` to look up whether each table actually has a column at that position.

In [0]:
-- Step 4: Map every table's columns to the unified list
-- CROSS JOIN pairs every table with every unified column (2 tables × 7 columns = 14 rows)
-- LEFT JOIN checks if the table actually has a column at that position

CREATE OR REPLACE TEMP VIEW table_column_map AS
SELECT
  t.table_name
  ,u.ordinal_position
  ,u.unified_col
  ,sc.column_name as actual_col
  ,CASE
    WHEN sc.column_name IS NULL               THEN concat('NULL AS ', u.unified_col)
    WHEN sc.column_name = u.unified_col       THEN sc.column_name
    ELSE concat(sc.column_name, ' AS ', u.unified_col)
  END as expression
FROM snapshot_tables t
CROSS JOIN unified_columns u
LEFT JOIN snapshot_columns sc
  ON t.table_name = sc.table_name
  AND u.ordinal_position = sc.ordinal_position
ORDER BY t.table_name, u.ordinal_position;

SELECT * FROM table_column_map;

Each row represents **one column for one table**. The `expression` column shows exactly what will appear in that table's `SELECT`:

* `school_urn` — column name matches the unified name, no change needed
* `urn AS school_urn` — column exists but with a different name, needs renaming
* `NULL AS region` — column doesn’t exist in this table, fill the gap with `NULL`

Now we need to combine these expressions into a complete `SELECT` statement for each table.

In [0]:
-- Step 5: For each table, combine the column expressions into a complete SELECT
-- This uses the same collect_list + aggregate pattern from earlier
-- but now runs once per table (GROUP BY table_name)

DECLARE OR REPLACE VARIABLE q STRING DEFAULT char(39);

CREATE OR REPLACE TEMP VIEW table_selects AS
SELECT
  table_name
  ,concat(
    'SELECT '
    ,aggregate(
      collect_list(expression)
      ,''
      ,(acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ', ', x) END
    )
    ,', ', q, table_name, q, ' AS source_table'
    ,' FROM catalog_40_copper_analyst_training.messy_data.', table_name
  ) as select_sql
FROM table_column_map
GROUP BY table_name
ORDER BY table_name;

SELECT * FROM table_selects;

Each row now contains a complete `SELECT ... FROM ...` statement for one table, with all columns mapped to the unified names. The `source_table` column identifies which table each row came from — the equivalent of `snapshot_term` in the earlier example.

The final step is to join these with `UNION ALL` and execute. We use `aggregate` again — this time to combine the per-table SELECTs with `UNION ALL` between each one.

In [0]:
-- Step 6: Join all SELECTs with UNION ALL, create view, and verify

DECLARE OR REPLACE VARIABLE full_query STRING;

SET VAR full_query = (
  SELECT concat(
    'CREATE OR REPLACE TEMP VIEW all_schools_combined AS '
    ,aggregate(
      collect_list(select_sql)
      ,''
      ,(acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ' UNION ALL ', x) END
    )
  )
  FROM table_selects
);

SELECT full_query as generated_sql;

In [0]:
-- Execute the generated query and verify the results
EXECUTE IMMEDIATE full_query;

SELECT
  source_table
  ,COUNT(*) as row_count
FROM all_schools_combined
GROUP BY source_table
ORDER BY source_table;

The entire process — from discovering tables to producing the combined view — ran without a single table or column name being hard-coded. Add a new snapshot next term (`schools_summer_2025`), and the query will discover and include it automatically.

The pattern works in three layers:
1. **Discover** — find tables and columns from `INFORMATION_SCHEMA`
2. **Map** — pair every table’s columns to a unified list, filling gaps with `NULL`
3. **Generate and execute** — build the `UNION ALL` as a string and run it with `EXECUTE IMMEDIATE`

This is the same `collect_list` → `aggregate` → `EXECUTE IMMEDIATE` technique from the two-table example, applied twice: once to build each table’s column list, and once to join the tables together.